# ADNI Comprehensive Exploratory Data Analysis

This notebook provides an interactive environment for exploring the Alzheimer's Disease Neuroimaging Initiative (ADNI) dataset.

## Contents
1. [Setup and Data Loading](#1-setup-and-data-loading)
2. [Schema Analysis](#2-schema-analysis)
3. [Statistical Analysis](#3-statistical-analysis)
4. [Longitudinal Analysis](#4-longitudinal-analysis)
5. [Multimodal Correlation Analysis](#5-multimodal-correlation-analysis)
6. [Machine Learning Readiness](#6-machine-learning-readiness)
7. [Visualizations](#7-visualizations)

## 1. Setup and Data Loading

In [ ]:
# Add project root to path
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

# Import modules
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.loader import load_and_preprocess_adni_data
from src.analysis.schema_analyzer import SchemaAnalyzer
from src.analysis.statistical_analyzer import StatisticalAnalyzer
from src.analysis.longitudinal_analyzer import LongitudinalAnalyzer
from src.analysis.ml_readiness import MLReadinessAnalyzer
from src.visualization.plots import ADNIVisualizer

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

# Set plot style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)

print("✓ Setup complete")

In [ ]:
# Load all datasets
datasets, summary_df = load_and_preprocess_adni_data()

print(f"Loaded {len(datasets)} datasets")
print("\nDataset Summary:")
summary_df

## 2. Schema Analysis

In [ ]:
# Initialize schema analyzer
schema_analyzer = SchemaAnalyzer(datasets)
schema_info = schema_analyzer.analyze_all()

# Get summary table
schema_df = schema_analyzer.get_summary_table()
schema_df

In [ ]:
# Join key availability
join_analysis = schema_analyzer.get_join_analysis()
join_analysis

In [ ]:
# Category aggregates
category_aggregates = schema_analyzer.get_category_aggregates()
category_aggregates

## 3. Statistical Analysis

In [ ]:
# Initialize statistical analyzer
stat_analyzer = StatisticalAnalyzer(datasets)

# Biomarker analysis
bio_results = stat_analyzer.analyze_biomarkers()

print(f"Dataset: {bio_results['dataset']}")
print(f"Total samples: {bio_results['total_samples']:,}")
print(f"Unique participants: {bio_results['unique_participants']:,}")

print("\nDescriptive Statistics:")
bio_results['descriptive_stats'].round(4)

In [ ]:
# Outlier analysis
print("Outlier Analysis (IQR 1.5× Rule):")
bio_results['outliers_iqr'].round(4)

In [ ]:
# Cognitive assessment analysis
cog_results = stat_analyzer.analyze_cognitive_assessments()

for name, result in cog_results.items():
    print(f"\n{name}:")
    print(f"  Total assessments: {result['total_assessments']:,}")
    print(f"  Unique participants: {result['unique_participants']:,}")
    display(result['descriptive_stats'].round(2))

In [ ]:
# Imaging analysis
img_results = stat_analyzer.analyze_imaging()

print(f"Dataset: {img_results['dataset']}")
print(f"Total scans: {img_results['total_scans']:,}")
print(f"Unique participants: {img_results['unique_participants']:,}")

print("\nDescriptive Statistics:")
img_results['descriptive_stats'].round(2)

## 4. Longitudinal Analysis

In [ ]:
# Initialize longitudinal analyzer
long_analyzer = LongitudinalAnalyzer(datasets)

# Visit patterns
visit_summary = long_analyzer.get_visit_summary_table()
visit_summary

In [ ]:
# Study phase distribution
phase_dist = long_analyzer.get_study_phase_distribution()
phase_dist

In [ ]:
# Temporal coverage
temporal = long_analyzer.analyze_temporal_coverage()
temporal

## 5. Multimodal Correlation Analysis

In [ ]:
# Multimodal correlations
corr_results = stat_analyzer.multimodal_correlation_analysis()

if 'biomarker_cognitive' in corr_results:
    bc = corr_results['biomarker_cognitive']
    print(f"Biomarker-Cognitive Correlations (n={bc['n_records']:,} records):")
    display(bc['correlation_matrix'].round(3))

In [ ]:
if 'biomarker_imaging' in corr_results:
    bi = corr_results['biomarker_imaging']
    print(f"Biomarker-Imaging Correlations (n={bi['n_participants']:,} participants):")
    display(bi['correlation_matrix'].round(3))

## 6. Machine Learning Readiness

In [ ]:
# Initialize ML readiness analyzer
ml_analyzer = MLReadinessAnalyzer(datasets)
ml_results = ml_analyzer.full_ml_assessment()

# Feature variance
print("Feature Variance Analysis:")
ml_results['feature_variance'].round(4)

In [ ]:
# Multicollinearity
print("Multicollinearity Analysis:")
if 'correlation_matrix' in ml_results['multicollinearity']:
    display(ml_results['multicollinearity']['correlation_matrix'].round(3))
    
    if ml_results['multicollinearity']['n_high_correlations'] > 0:
        print("\nHigh Correlation Pairs (|r| > 0.7):")
        display(ml_results['multicollinearity']['high_correlation_pairs'])

In [ ]:
# Missing data pattern
print("Missing Data Pattern:")
ml_results['missing_data']

In [ ]:
# Complete cases
cc = ml_results['complete_cases']
print(f"Total Records: {cc['total_records']:,}")
print(f"Complete Cases: {cc['complete_cases']:,} ({cc['complete_cases_pct']:.1f}%)")
print(f"Has Biomarker: {cc['has_biomarker']:,}")
print(f"Has Cognitive: {cc['has_cognitive']:,}")
print(f"Has Both: {cc['has_both']:,}")

In [ ]:
# Target variable analysis
ta = ml_results['target_analysis']
print(f"Target: {ta['target_column']}")
print(f"Valid Samples: {ta['n_valid']:,}")
print(f"\nDistribution: {ta['distribution']}")

if ta['binary_targets']:
    print("\nBinary Targets:")
    for name, info in ta['binary_targets'].items():
        print(f"  {name}: {info['n']:,} ({info['pct']:.1f}%)")

if ta['imbalance_ratio']:
    print(f"\nClass Imbalance Ratio: {ta['imbalance_ratio']:.2f}:1")

## 7. Visualizations

In [ ]:
# Initialize visualizer
from config.settings import VISUALIZATIONS_DIR
visualizer = ADNIVisualizer(VISUALIZATIONS_DIR)

# Dataset overview
visualizer.plot_dataset_overview(summary_df, '01_dataset_overview.png')

In [ ]:
# Missing data
visualizer.plot_missing_data(schema_info, '02_missing_data.png')

In [ ]:
# Biomarker distributions
upenn_df = datasets['UPENN_PLASMA_FUJIREBIO_QUANTERIX_13Feb2026']['df']
visualizer.plot_biomarker_distributions(upenn_df, '03_biomarker_distributions.png')

In [ ]:
# Correlation heatmap
if 'biomarker_cognitive' in corr_results:
    corr_matrix = corr_results['biomarker_cognitive']['correlation_matrix']
    visualizer.plot_correlation_heatmap(corr_matrix, '04_correlation_heatmap.png')

In [ ]:
# Study phases
if not phase_dist.empty:
    visualizer.plot_study_phases(phase_dist, '05_study_phases.png')

In [ ]:
# Longitudinal patterns
visit_summaries = long_analyzer.analyze_all_datasets()
visualizer.plot_longitudinal_patterns(visit_summaries, '06_longitudinal_patterns.png')

In [ ]:
# CDR distribution
cdr_df = datasets['CDR_13Feb2026']['df']
cdr_global = cdr_df['CDGLOBAL'].dropna()
cdr_global = cdr_global[cdr_global >= 0]
cdr_dist = cdr_global.value_counts().sort_index().to_dict()
visualizer.plot_cdr_distribution(cdr_dist, '07_cdr_distribution.png')

---

## Summary

This notebook has demonstrated:

1. **Data Loading**: Loaded and preprocessed 29 ADNI datasets
2. **Schema Analysis**: Analyzed table relationships and join keys
3. **Statistical Analysis**: Computed descriptive statistics for biomarkers, cognitive assessments, and imaging
4. **Longitudinal Analysis**: Examined visit patterns and temporal coverage
5. **Multimodal Correlations**: Analyzed relationships between biomarkers, cognitive scores, and imaging
6. **ML Readiness**: Assessed data suitability for machine learning models
7. **Visualizations**: Generated publication-quality figures

For batch processing, use the `run_eda.py` script from the command line.